In [1]:
import pandas as pd

from environment.sae_warning_env import SAEWarningEnv


df = pd.read_csv(
    "../data/processed/mimic3/sae_rl_dataset.csv"
)

print("Dataset:", df.shape)

env = SAEWarningEnv(df)

print("Environment created")
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

state, info = env.reset(seed=42)

print("\nInitial state:")
print(state)

print("\nInfo:")
print(info)

next_state, reward, terminated, truncated, info = env.step(0)

print("\nAfter action 0:")
print("Next state:", next_state)
print("Reward:", reward)
print("Terminated:", terminated)
print("Info:", info)

ModuleNotFoundError: No module named 'environment'

In [ ]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

Current directory:
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/test

Files/folders here:
['test1.ipynb', 'test.ipynb']


In [ ]:
from pathlib import Path

project_root = Path.cwd().parent

print("Project root:", project_root)
print("Environment exists:", (project_root / "environment").exists())
print(
    "RL environment file exists:",
    (project_root / "environment" / "sae_warning_env.py").exists()
)

Project root: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning
Environment exists: True
RL environment file exists: True


In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

sys.path.insert(0, str(project_root))

from environment.sae_warning_env import SAEWarningEnv

print("SAEWarningEnv imported successfully!")

SAEWarningEnv imported successfully!


In [3]:
import pandas as pd

df = pd.read_csv(
    project_root / "data/processed/mimic3/sae_rl_dataset.csv"
)

print("Dataset:", df.shape)

env = SAEWarningEnv(df)

print("Environment created")
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

state, info = env.reset(seed=42)

print("\nInitial state:")
print(state)

print("\nInfo:")
print(info)

next_state, reward, terminated, truncated, info = env.step(0)

print("\nAfter action 0:")
print("Next state:", next_state)
print("Reward:", reward)
print("Terminated:", terminated)
print("Info:", info)

Dataset: (4556, 12)
Environment created
Observation space: Box(-inf, inf, (7,), float32)
Action space: Discrete(3)

Initial state:
[ 15.   10.  101.5  69.   21.   98.    1. ]

Info:
{'icustay_id': np.int64(206504), 'hour': 1}

After action 0:
Next state: [15. 15. 96. 69. 20. 98.  2.]
Reward: 1.0
Terminated: False
Info: {'icustay_id': np.int64(206504), 'hour': 1, 'sae': 0, 'action': 0}


In [4]:
# ============================================================
# CREATE FUTURE SAE TARGET
# ============================================================

import pandas as pd

df = pd.read_csv(
    "../data/processed/mimic3/sae_rl_dataset.csv"
)

df = df.sort_values(
    ["icustay_id", "hour"]
).reset_index(drop=True)

# SAE occurring at the NEXT hourly observation
df["future_sae_1h"] = (
    df.groupby("icustay_id")["sae"]
      .shift(-1)
      .fillna(0)
      .astype(int)
)

print("Dataset shape:", df.shape)

print("\nCurrent SAE:")
print(df["sae"].value_counts())

print("\nFuture 1-hour SAE:")
print(df["future_sae_1h"].value_counts())

print("\nFuture SAE events:")
print(df["future_sae_1h"].sum())

Dataset shape: (4556, 13)

Current SAE:
sae
0    4521
1      35
Name: count, dtype: int64

Future 1-hour SAE:
future_sae_1h
0    4521
1      35
Name: count, dtype: int64

Future SAE events:
35


In [5]:
# ============================================================
# VERIFY TEMPORAL SHIFT
# ============================================================

display(
    df[
        [
            "icustay_id",
            "hour",
            "gcs_last_observed",
            "previous_observed_gcs",
            "sae",
            "future_sae_1h"
        ]
    ]
    .loc[lambda x: x["future_sae_1h"] == 1]
    .head(20)
)

,icustay_id,hour,gcs_last_observed,previous_observed_gcs,sae,future_sae_1h
10,201006,10,15.0,15.0,0,1
15,201006,15,8.0,8.0,0,1
168,203766,11,15.0,15.0,0,1
288,203766,139,9.0,9.0,0,1
289,203766,140,6.0,9.0,1,1
319,205170,26,14.0,14.0,0,1
355,205170,62,15.0,15.0,0,1
427,209797,44,11.0,11.0,0,1
696,222779,53,15.0,15.0,0,1
760,223177,35,11.0,11.0,0,1


In [6]:
# ============================================================
# PATIENT-LEVEL TRAIN / TEST SPLIT
# ============================================================

import numpy as np
import pandas as pd

# Unique patients
patients = df["subject_id"].unique()

print("Total patients:", len(patients))

# Reproducible shuffle
rng = np.random.default_rng(42)
rng.shuffle(patients)

# 80/20 split
split_index = int(len(patients) * 0.8)

train_patients = patients[:split_index]
test_patients = patients[split_index:]

train_df = df[
    df["subject_id"].isin(train_patients)
].copy()

test_df = df[
    df["subject_id"].isin(test_patients)
].copy()

print("\nTRAIN")
print("Patients:", train_df["subject_id"].nunique())
print("ICU stays:", train_df["icustay_id"].nunique())
print("Rows:", len(train_df))
print("Future SAE events:", train_df["future_sae_1h"].sum())

print("\nTEST")
print("Patients:", test_df["subject_id"].nunique())
print("ICU stays:", test_df["icustay_id"].nunique())
print("Rows:", len(test_df))
print("Future SAE events:", test_df["future_sae_1h"].sum())

print("\nPatient overlap:")
print(
    len(
        set(train_patients)
        &
        set(test_patients)
    )
)

Total patients: 25

TRAIN
Patients: 20
ICU stays: 33
Rows: 3302
Future SAE events: 22

TEST
Patients: 5
ICU stays: 5
Rows: 1254
Future SAE events: 13

Patient overlap:
0


In [7]:
# ============================================================
# SAVE PATIENT-LEVEL TRAIN / TEST DATA
# ============================================================

train_path = "../data/processed/mimic3/sae_rl_train.csv"
test_path = "../data/processed/mimic3/sae_rl_test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("Saved training dataset:", train_path)
print("Saved testing dataset:", test_path)

print("\nTrain shape:", train_df.shape)
print("Test shape:", test_df.shape)

Saved training dataset: ../data/processed/mimic3/sae_rl_train.csv
Saved testing dataset: ../data/processed/mimic3/sae_rl_test.csv

Train shape: (3302, 13)
Test shape: (1254, 13)
